# Task 3: Multi-Head Attention

Multi-head attention allows the model to attend to different representation subspaces simultaneously.

## What You'll Learn
1. **Why Multiple Heads?** - Different heads learn different relationships
2. **Multi-Head Implementation** - Parallel attention computation
3. **Concatenation & Projection** - Combining head outputs

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Theory: Multi-Head Attention

Multi-head attention applies $h$ attention heads in parallel:

**MultiHead(Q,K,V) = Concat(head_1, ..., head_h)W^O**

where **head_i = Attention(QW_i^Q, KW_i^K, VW_i^V)**

Each head can learn different types of relationships!

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    """Multi-head attention mechanism."""
    
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.d_k = embed_dim // num_heads  # dimension per head
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        # Linear projections for Q, K, V (all heads at once)
        self.W_q = torch.nn.Linear(embed_dim, embed_dim)
        self.W_k = torch.nn.Linear(embed_dim, embed_dim)
        self.W_v = torch.nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.W_o = torch.nn.Linear(embed_dim, embed_dim)
        
        self.dropout = torch.nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project and reshape for multiple heads
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        context = torch.matmul(attn_weights, V)
        
        # Concatenate heads: (batch, num_heads, seq_len, d_k) -> (batch, seq_len, embed_dim)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        
        output = self.W_o(context)
        
        return output, attn_weights

# Test
mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x = torch.randn(2, 10, 64)
output, weights = mha(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Attention weights: {weights.shape}  (batch, heads, seq, seq)")

In [ ]:
# Visualize different attention heads
mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x = torch.randn(1, 12, 64)
output, weights = mha(x)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i in range(8):
    axes[i].imshow(weights[0, i].detach().numpy(), cmap='viridis')
    axes[i].set_title(f'Head {i+1}')
    axes[i].set_xlabel('Position')
    axes[i].set_ylabel('Position')

plt.suptitle('Different Attention Heads Learn Different Patterns', y=1.02)
plt.tight_layout()
plt.show()

## Exercises

### Exercise 1: Implement Multi-Head from Single-Head
Build multi-head attention by combining multiple SingleAttention instances.

In [ ]:
# SOLUTION: Multi-head from single-head
class MultiHeadFromSingle(torch.nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.d_k = embed_dim // num_heads
        self.num_heads = num_heads
        
        # Multiple single-head attention layers
        self.heads = torch.nn.ModuleList([
            SelfAttention(embed_dim, self.d_k, self.d_k)
            for _ in range(num_heads)
        ])
        
        self.W_o = torch.nn.Linear(embed_dim, embed_dim)
    
    def forward(self, x):
        # Apply each head
        head_outputs = [head(x) for head in self.heads]
        
        # Concatenate outputs
        concatenated = torch.cat([out[0] for out in head_outputs], dim=-1)
        
        # Project
        return self.W_o(concatenated)

# Helper: SelfAttention from Task 2
class SelfAttention(torch.nn.Module):
    def __init__(self, embed_dim, d_k, d_v):
        super().__init__()
        self.W_q = torch.nn.Linear(embed_dim, d_k)
        self.W_k = torch.nn.Linear(embed_dim, d_k)
        self.W_v = torch.nn.Linear(embed_dim, d_v)
    
    def forward(self, x):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(Q.size(-1))
        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, V), weights

mha = MultiHeadFromSingle(64, 8)
out = mha(x)
print(f"Output shape: {out.shape}")

## Summary
- ✅ Multiple heads learn different attention patterns
- ✅ Concatenation followed by linear projection
- ✅ Parallel computation is efficient

## Next: [Task 4: Transformer Encoder](../task-04-transformer-encoder/overview.html)